# Notebook 03 — Evaluate All Experiments
**Member: C** | Set `model.checkpoint_path` in `configs/eval.yaml` before each run.

| Experiment | `checkpoint_path` | Benchmark |
|---|---|---|
| Exp 0 zero-shot    | `null`                             | GSM8K |
| Exp 1 few-shot CoT | `null`                             | GSM8K |
| Exp 2 adaptation   | `results/checkpoints/adaptation`   | GSM8K |
| Exp 3 distillation | `results/checkpoints/distillation` | GSM8K |
| Exp 4 filtered     | `results/checkpoints/filtered`     | GSM8K |
| Exp 5 cross-task   | `results/checkpoints/filtered`     | ARC |

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:

from pathlib import Path
import subprocess

REPO = Path("/content/SC4001-Group133")

# Download only if the repository is not already present
if REPO.exists():
    assert (REPO / ".git").is_dir(), \
        "Folder exists but is not a Git repository. Stop here."
    print("Existing GitHub project found.")
else:
    subprocess.run([
        "git", "clone",
        "https://github.com/TIFFMUN/SC4001-Group133.git",
        str(REPO)
    ], check=True)
    print("GitHub project downloaded successfully!")

# Record the exact code version
commit = subprocess.check_output([
    "git", "-C", str(REPO),
    "rev-parse", "HEAD"
], text=True).strip()

print("Repository:", REPO)
print("Commit:", commit)
print("Evaluation files:")
print(list((REPO / "src/evaluation").glob("*.py")))


GitHub project downloaded successfully!
Repository: /content/SC4001-Group133
Commit: 2e7a137990cfcb21f4400964b1d00cb1ae37b9d8
Evaluation files:
[PosixPath('/content/SC4001-Group133/src/evaluation/metrics.py'), PosixPath('/content/SC4001-Group133/src/evaluation/budget_router.py'), PosixPath('/content/SC4001-Group133/src/evaluation/eval_arc.py'), PosixPath('/content/SC4001-Group133/src/evaluation/eval_router.py'), PosixPath('/content/SC4001-Group133/src/evaluation/__init__.py'), PosixPath('/content/SC4001-Group133/src/evaluation/eval_gsm8k.py')]


In [15]:

from pathlib import Path

repo = Path("/content/SC4001-Group133")
setup_file = repo / "src/setup_cache.py"

if setup_file.is_file():
    print("=== SETUP FUNCTION SOURCE ===")
    print(setup_file.read_text(encoding="utf-8"))
else:
    print("setup_cache.py not found")
    print("Matching files:")
    for p in repo.rglob("*setup_cache*"):
        print(p)


=== SETUP FUNCTION SOURCE ===
"""
setup_cache.py
--------------
Must be called FIRST in every Colab notebook before any HuggingFace imports.

Redirects the HuggingFace cache (datasets + models) to Google Drive so that:
  - Datasets are downloaded once and reused across sessions
  - Model weights are cached and not re-downloaded each time
  - Distillation traces and processed datasets persist

Usage (first cell of every notebook):
    import sys
    sys.path.insert(0, '/content/SC4001-Group133')
    from src.setup_cache import setup

    setup()   # call before ANY import from transformers/datasets/peft

If not running on Colab (local or Docker), this is a no-op.
"""

import os
import logging

logger = logging.getLogger(__name__)

# TODO: change this to match your actual Google Drive folder structure
DRIVE_ROOT = "/content/drive/MyDrive/SC4001"


def setup(drive_root: str = DRIVE_ROOT) -> None:
    """
    Point all HuggingFace caches to Google Drive.
    Must be called before importing

In [16]:

from pathlib import Path

repo = Path("/content/SC4001-Group133")
drive_root = Path("/content/drive/MyDrive/SC4001")

print("=== LOCAL FOLDERS ===")

for name in ["data", "results"]:
    path = repo / name
    print(f"\n{name}:")
    print("Exists:", path.exists())
    print("Symlink:", path.is_symlink())

    if path.is_dir() and not path.is_symlink():
        print("Contents:", [p.name for p in path.iterdir()][:10])

print("\n=== GOOGLE DRIVE FOLDERS ===")

for name in ["data", "results"]:
    path = drive_root / name
    print(f"{name}:", "FOUND" if path.is_dir() else "NOT FOUND")

print("\nCheck complete. Nothing was modified.")


=== LOCAL FOLDERS ===

data:
Exists: True
Symlink: False
Contents: ['exp3_exp4_distillation_data.zip', 'README.md']

results:
Exists: True
Symlink: False
Contents: ['exp6_router_tall_hard_gsm8k.csv', 'plots', 'exp2_adaptation_r16_gsm8k.csv', 'exp6_router_sweep.csv', 'exp2_adaptation_r16_500_gsm8k.csv', 'exp6_router_tall_easy_gsm8k.csv', '.gitkeep', 'exp3_distillation_updated_500_600tokens_gsm8k.csv', 'exp5_d3_cross_task_arc.csv', 'exp0_zero_shot_gsm8k.csv']

=== GOOGLE DRIVE FOLDERS ===
data: FOUND
results: FOUND

Check complete. Nothing was modified.


In [17]:

import os
import sys
import torch
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

REPO = Path("/content/SC4001-Group133")
assert (REPO / "src/baselines/cot_prompting.py").is_file(), \
    "GitHub project not found!"

os.chdir(REPO)
sys.path.insert(0, str(REPO))

# New folder ONLY for Experiment 1d
OUT_DIR = Path(
    "/content/drive/MyDrive/SC4001/results/exp1d_concise_500_run"
)

assert not OUT_DIR.exists(), \
    "STOP! Output folder already exists. Do not overwrite."

assert torch.cuda.is_available(), \
    "GPU unavailable. Enable T4 GPU first."

OUT_DIR.mkdir(parents=True, exist_ok=False)

print("GPU:", torch.cuda.get_device_name(0))
print("Output folder:", OUT_DIR)
print("SAFE SETUP COMPLETE!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
Output folder: /content/drive/MyDrive/SC4001/results/exp1d_concise_500_run
SAFE SETUP COMPLETE!


In [18]:

# ============================================================
# EXPERIMENT 1D — CONCISE PROMPTING (500 GSM8K QUESTIONS)
# Original Qwen2.5-0.5B-Instruct model, NO LoRA
# Saves CSV and JSON separately without overwriting results
# ============================================================

import os
import sys
import json
import torch
import shutil
import tempfile
from pathlib import Path
from tqdm import tqdm

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.baselines.cot_prompting import build_concise_prompt
from src.distillation.filter_traces import extract_answer, answers_match
from src.evaluation.metrics import (
    compute_latency,
    compute_summary_stats,
    save_results,
    print_summary
)

# ------------------------------------------------------------
# 1. Experiment settings
# ------------------------------------------------------------

MAX_SAMPLES = 500
MAX_NEW_TOKENS = 600
EXPERIMENT_NAME = "exp1d_concise_500"
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

OUT_DIR = Path(
    "/content/drive/MyDrive/SC4001/results/exp1d_concise_500_run"
)

CSV_PATH = OUT_DIR / f"{EXPERIMENT_NAME}_gsm8k.csv"
JSON_PATH = OUT_DIR / f"{EXPERIMENT_NAME}_gsm8k_summary.json"

# ------------------------------------------------------------
# 2. Safety checks
# ------------------------------------------------------------

assert torch.cuda.is_available(), "GPU not enabled!"
assert OUT_DIR.is_dir(), "Output folder missing. Run safe setup first."

assert not CSV_PATH.exists(), \
    "STOP: CSV already exists. Refusing to overwrite."

assert not JSON_PATH.exists(), \
    "STOP: JSON already exists. Refusing to overwrite."

print("=== SAFETY CHECK PASSED ===")
print("Experiment:", EXPERIMENT_NAME)
print("Samples:", MAX_SAMPLES)
print("Max new tokens:", MAX_NEW_TOKENS)
print("Output folder:", OUT_DIR)
print("Previous results will not be overwritten.")

# ------------------------------------------------------------
# 3. Load ORIGINAL pretrained model (NO LoRA)
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("\nOriginal pretrained model loaded.")

# ------------------------------------------------------------
# 4. Load first 500 GSM8K test questions
# ------------------------------------------------------------

test_data = load_dataset(
    "openai/gsm8k",
    "main"
)["test"].select(range(MAX_SAMPLES))

print("Test questions loaded:", len(test_data))

# ------------------------------------------------------------
# 5. Run ONLY Experiment 1d — Concise Prompting
# ------------------------------------------------------------

results = []

with torch.no_grad():
    for sample in tqdm(test_data, desc=EXPERIMENT_NAME):

        prompt = build_concise_prompt(
            sample["question"],
            tokenizer
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(model.device)

        input_len = inputs["input_ids"].shape[1]

        latency_ms, output_ids = compute_latency(
            model=model,
            inputs=inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False
        )

        generated_ids = output_ids[0][input_len:]

        prediction = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        )

        predicted_answer = extract_answer(prediction)

        is_correct = answers_match(
            predicted_answer,
            sample["answer"]
        )

        results.append({
            "question": sample["question"],
            "ground_truth": sample["answer"],
            "prediction": prediction,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct,
            "latency_ms": latency_ms,
            "token_count": len(generated_ids)
        })

# ------------------------------------------------------------
# 6. Calculate evaluation metrics
# ------------------------------------------------------------

assert len(results) == 500, \
    "Expected 500 evaluated questions."

stats = compute_summary_stats(results)

print("\n=== EXPERIMENT 1D RESULTS ===")
print_summary(EXPERIMENT_NAME, stats)

# ------------------------------------------------------------
# 7. Save CSV and JSON WITHOUT overwriting existing files
# ------------------------------------------------------------

# Recheck before saving
assert not CSV_PATH.exists(), "CSV already exists!"
assert not JSON_PATH.exists(), "JSON already exists!"

# Write CSV to a temporary local folder first
with tempfile.TemporaryDirectory() as temp_dir:

    temp_csv = Path(temp_dir) / CSV_PATH.name

    save_results(results, str(temp_csv))

    # Exclusive creation: fails if destination exists
    with temp_csv.open("rb") as src:
        with CSV_PATH.open("xb") as dst:
            shutil.copyfileobj(src, dst)

# Save JSON with exclusive creation
with JSON_PATH.open("x") as f:
    json.dump(
        {"experiment": EXPERIMENT_NAME, **stats},
        f,
        indent=2
    )

print("\n=== FILES SAVED SUCCESSFULLY ===")
print("CSV:", CSV_PATH)
print("JSON:", JSON_PATH)

print("\n=== FINAL RESULTS ===")
print("Accuracy:", f"{stats['accuracy']:.2%}")
print("Mean tokens:", round(stats["mean_token_count"], 2))
print("Mean latency (ms):", round(stats["mean_latency_ms"], 2))

print("\nEXPERIMENT 1D COMPLETE!")


=== SAFETY CHECK PASSED ===
Experiment: exp1d_concise_500
Samples: 500
Max new tokens: 600
Output folder: /content/drive/MyDrive/SC4001/results/exp1d_concise_500_run
Previous results will not be overwritten.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Original pretrained model loaded.


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (3 outputs left out)

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Test questions loaded: 500


exp1d_concise_500: 100%|██████████| 500/500 [54:38<00:00,  6.56s/it]


=== EXPERIMENT 1D RESULTS ===

=== FILES SAVED SUCCESSFULLY ===
CSV: /content/drive/MyDrive/SC4001/results/exp1d_concise_500_run/exp1d_concise_500_gsm8k.csv
JSON: /content/drive/MyDrive/SC4001/results/exp1d_concise_500_run/exp1d_concise_500_gsm8k_summary.json

=== FINAL RESULTS ===
Accuracy: 18.20%
Mean tokens: 218.73
Mean latency (ms): 6553.9

EXPERIMENT 1D COMPLETE!


In [19]:

import pandas as pd
import json
from pathlib import Path

OUT_DIR = Path(
    "/content/drive/MyDrive/SC4001/results/exp1d_concise_500_run"
)

csv_path = OUT_DIR / "exp1d_concise_500_gsm8k.csv"
json_path = OUT_DIR / "exp1d_concise_500_gsm8k_summary.json"

assert csv_path.exists(), "CSV missing!"
assert json_path.exists(), "JSON missing!"

df = pd.read_csv(csv_path)

with open(json_path) as f:
    stats = json.load(f)

assert len(df) == 500, f"Expected 500 questions, found {len(df)}"

print("=== EXPERIMENT 1D VERIFIED ===")
print("Questions evaluated:", len(df))
print("Correct answers:", df["is_correct"].sum())
print("Accuracy:", f"{stats['accuracy']:.2%}")
print("Mean tokens:", round(stats["mean_token_count"], 2))
print("Mean latency (ms):", round(stats["mean_latency_ms"], 2))

print("\nCSV:", csv_path)
print("JSON:", json_path)
print("\nVerification successful!")


=== EXPERIMENT 1D VERIFIED ===
Questions evaluated: 500
Correct answers: 91
Accuracy: 18.20%
Mean tokens: 218.73
Mean latency (ms): 6553.9

CSV: /content/drive/MyDrive/SC4001/results/exp1d_concise_500_run/exp1d_concise_500_gsm8k.csv
JSON: /content/drive/MyDrive/SC4001/results/exp1d_concise_500_run/exp1d_concise_500_gsm8k_summary.json

Verification successful!


In [7]:
from pathlib import Path
import pandas as pd
import json

results = Path("/content/drive/MyDrive/SC4001/results")

csv_path = results / "exp1e_format_reinforced_gsm8k.csv"
json_path = results / "summary_results/exp1e_format_reinforced_gsm8k_summary.json"

print("CSV saved:", csv_path.is_file())
print("JSON saved:", json_path.is_file())

if csv_path.is_file():
    df = pd.read_csv(csv_path)
    print("Questions evaluated:", len(df))

if json_path.is_file():
    print(json.dumps(json.loads(json_path.read_text()), indent=2))

CSV saved: True
JSON saved: True
Questions evaluated: 500
{
  "experiment": "exp1e_format_reinforced",
  "accuracy": 0.446,
  "mean_latency_ms": 9280.99,
  "median_latency_ms": 8732.64,
  "mean_token_count": 315.58,
  "n_samples": 500
}


In [8]:
import pandas as pd
from pathlib import Path

results_dir = Path("/content/drive/MyDrive/SC4001/results")

experiments = {
    "Exp 3 - Distilled": "exp3_distillation_updated_500_600tokens_gsm8k.csv",
    "Exp 4 - Filtered": "exp4_filtered_updated_500_600tokens_gsm8k.csv",
    "Exp 1e - Format Reinforced": "exp1e_format_reinforced_gsm8k.csv",
}

for name, filename in experiments.items():
    path = results_dir / filename

    if not path.exists():
        print(f"{name}: File not found")
        continue

    df = pd.read_csv(path)

    print(f"\n{name}")
    print("Total questions:", len(df))
    print("First 100 accuracy:", f"{df.head(100)['is_correct'].mean():.2%}")
    print("Full accuracy:", f"{df['is_correct'].mean():.2%}")


Exp 3 - Distilled
Total questions: 500
First 100 accuracy: 41.00%
Full accuracy: 40.80%

Exp 4 - Filtered
Total questions: 500
First 100 accuracy: 34.00%
Full accuracy: 35.80%

Exp 1e - Format Reinforced
Total questions: 500
First 100 accuracy: 40.00%
Full accuracy: 44.60%


In [20]:

from pathlib import Path

REPO = Path("/content/SC4001-Group133")

assert REPO.is_dir(), "Repository missing!"

print("=== BASELINE FILES ===")
for p in sorted((REPO / "src/baselines").glob("*.py")):
    print(p.name)

for filename in [
    "src/evaluation/eval_gsm8k.py",
    "src/baselines/zero_shot.py"
]:
    path = REPO / filename
    if path.is_file():
        print(f"\n=== {filename} ===")
        print(path.read_text(encoding="utf-8"))
    else:
        print(f"\nNot found: {filename}")

print("\nCHECK COMPLETE — no experiment was run.")


=== BASELINE FILES ===
__init__.py
cot_prompting.py

=== src/evaluation/eval_gsm8k.py ===
"""
eval_gsm8k.py
-------------
Evaluates a model (base or LoRA-finetuned) on the GSM8K test set.

Used for: Exp 0, 1, 2, 3, 4 (all experiments on the math benchmark).
Exp 5 (cross-task transfer) uses eval_arc.py on the Exp 4 checkpoint.

Answer parsing:
    GSM8K ground truth always contains '#### <number>'.
    We extract the predicted answer using the same pattern from the model output.
    This is the same logic as filter_traces.py — kept consistent on purpose.

Member responsible: C
"""

import logging
import os
from pathlib import Path

import torch
import yaml
from datasets import load_dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

from src.distillation.filter_traces import extract_answer, answers_match
from src.evaluation.metrics import compute_latency, compute_summary_stats, save_results, print_summary
from src.baseli

In [21]:

from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/SC4001")
OUT_DIR = DRIVE / "results" / "exp0_100_updated_run"

print("=== SAFETY CHECK ===")

assert DRIVE.is_dir(), "Google Drive not mounted!"
assert not OUT_DIR.exists(), (
    "STOP: Output folder already exists. "
    "Do not run the evaluation."
)

print("New experiment: exp0_100_updated")
print("New output folder:", OUT_DIR)
print("No existing folder with this name.")
print("SAFE TO PROCEED.")


=== SAFETY CHECK ===
New experiment: exp0_100_updated
New output folder: /content/drive/MyDrive/SC4001/results/exp0_100_updated_run
No existing folder with this name.
SAFE TO PROCEED.


In [22]:

# ============================================================
# FRESH EXPERIMENT 0 — 100 GSM8K QUESTIONS
# Zero-shot pretrained baseline, NO LoRA
# Uses the repository's original eval_gsm8k.py
# Saves into a NEW folder, never overwrites old results
# ============================================================

import os
import sys
import json
import yaml
import subprocess
from pathlib import Path

REPO = Path("/content/SC4001-Group133")
DRIVE = Path("/content/drive/MyDrive/SC4001")


EXPERIMENT = "exp0_100_updated"

OUT_DIR = DRIVE / "results" / "exp0_100_updated_run"


# 1. Check prerequisites
assert (REPO / "src/evaluation/eval_gsm8k.py").is_file(), \
    "Evaluation script not found."

assert (REPO / "configs/eval.yaml").is_file(), \
    "Evaluation configuration not found."

assert DRIVE.is_dir(), "Mount Google Drive first."

assert not OUT_DIR.exists(), \
    "STOP: This output folder already exists. Do not overwrite."

import torch
assert torch.cuda.is_available(), "Enable T4 GPU first."

# 2. Read ORIGINAL evaluation configuration
with open(REPO / "configs/eval.yaml") as f:
    cfg = yaml.safe_load(f)

assert "model" in cfg and "output" in cfg
assert cfg["model"]["base_model"] == "Qwen/Qwen2.5-0.5B-Instruct"

# Explicitly disable LoRA / fine-tuned checkpoints
cfg["model"]["checkpoint_path"] = None

# Write results ONLY to our new folder
cfg["output"]["results_dir"] = str(OUT_DIR)

# 3. Create separate output folder and configuration
OUT_DIR.mkdir(parents=True, exist_ok=False)

safe_config = OUT_DIR / "eval_exp0_100.yaml"

with safe_config.open("x") as f:
    yaml.safe_dump(cfg, f)

# 4. Run the ORIGINAL repository evaluator
# No --few_shot flag = zero-shot baseline
# No --checkpoint flag + null config = no LoRA

cmd = [
    sys.executable,
    "-m", "src.evaluation.eval_gsm8k",
    "--config", str(safe_config),
    "--experiment", EXPERIMENT,
    "--max_samples", "100",
    "--max_new_tokens", "600",
]

print("=== FRESH EXP 0 SETTINGS ===")
print("Model:", cfg["model"]["base_model"])
print("LoRA checkpoint:", cfg["model"]["checkpoint_path"])
print("Samples: 100")
print("Max new tokens: 600")
print("Few-shot: DISABLED")
print("Output:", OUT_DIR)

os.chdir(REPO)

subprocess.run(cmd, check=True)

# 5. Verify saved files
csv_path = OUT_DIR / f"{EXPERIMENT}_gsm8k.csv"
json_path = (
    OUT_DIR
    / "summary_results"
    / f"{EXPERIMENT}_gsm8k_summary.json"
)

assert csv_path.is_file(), "CSV not found!"
assert json_path.is_file(), "Summary JSON not found!"

import pandas as pd

df = pd.read_csv(csv_path)

assert len(df) == 100, \
    f"Expected 100 questions, got {len(df)}"

with json_path.open() as f:
    stats = json.load(f)

print("\n=== FRESH EXP 0 COMPLETE ===")
print("Questions:", len(df))
print("Accuracy:", f"{stats['accuracy']:.2%}")
print("Mean tokens:", round(stats["mean_token_count"], 2))

print("\nCSV:", csv_path)
print("JSON:", json_path)
print("\nPrevious experiment results were not overwritten.")


=== FRESH EXP 0 SETTINGS ===
Model: Qwen/Qwen2.5-0.5B-Instruct
LoRA checkpoint: None
Samples: 100
Max new tokens: 600
Few-shot: DISABLED
Output: /content/drive/MyDrive/SC4001/results/exp0_100_updated_run

=== FRESH EXP 0 COMPLETE ===
Questions: 100
Accuracy: 40.00%
Mean tokens: 306.8

CSV: /content/drive/MyDrive/SC4001/results/exp0_100_updated_run/exp0_100_updated_gsm8k.csv
JSON: /content/drive/MyDrive/SC4001/results/exp0_100_updated_run/summary_results/exp0_100_updated_gsm8k_summary.json

Previous experiment results were not overwritten.


In [23]:

import pandas as pd
import json
from pathlib import Path

OUT_DIR = Path(
    "/content/drive/MyDrive/SC4001/results/exp0_100_updated_run"
)

csv_path = OUT_DIR / "exp0_100_updated_gsm8k.csv"
json_path = OUT_DIR / "summary_results/exp0_100_updated_gsm8k_summary.json"

# Load results
df = pd.read_csv(csv_path)

# Calculate mean latency from the CSV
mean_latency_ms = df["latency_ms"].mean()
mean_latency_s = mean_latency_ms / 1000

print("=== EXP 0 BASELINE (100 SAMPLES) ===")
print("Total questions:", len(df))
print(f"Mean latency: {mean_latency_ms:.2f} ms")
print(f"Mean latency: {mean_latency_s:.2f} seconds")

# Compare against saved summary, if available
if json_path.exists():
    with open(json_path) as f:
        stats = json.load(f)

    print("\nSaved summary:")
    print(json.dumps(stats, indent=2))


=== EXP 0 BASELINE (100 SAMPLES) ===
Total questions: 100
Mean latency: 9349.60 ms
Mean latency: 9.35 seconds

Saved summary:
{
  "experiment": "exp0_100_updated",
  "accuracy": 0.4,
  "mean_latency_ms": 9349.6,
  "median_latency_ms": 8904.89,
  "mean_token_count": 306.8,
  "n_samples": 100
}


In [ ]:
# Exp 2-4 — set checkpoint_path in configs/eval.yaml, update EXPERIMENT_NAME, then run
EXPERIMENT_NAME = 'exp2_adaptation_r16'  # change per run
MAX_SAMPLES = 5  # set to 5 for quick test, 100 for full eval, None for all 1319

from src.evaluation.eval_gsm8k import evaluate_gsm8k
evaluate_gsm8k(experiment_name=EXPERIMENT_NAME, max_samples=MAX_SAMPLES)

In [ ]:
# Exp 5 — ARC cross-task transfer (uses Exp 4 checkpoint)
from src.evaluation.eval_arc import evaluate_arc
evaluate_arc(experiment_name='exp5_d3_cross_task')

In [ ]:
# List all saved CSVs on Drive
csvs = [f for f in os.listdir('/content/drive/MyDrive/SC4001/results') if f.endswith('.csv')]
for f in sorted(csvs):
    print(f)

## Exp 6 — Inference-gated budget router (optional, run after Exps 0–5)

Prerequisite: set `router.enabled: true` and `router.checkpoint_path` in `configs/eval.yaml`.

The router sweeps a range of complexity thresholds and saves a Pareto-ready CSV to
`results/exp6_router_sweep.csv`. Use `max_samples=50` for a quick sanity check first.

In [ ]:
# Exp 6 — budget router threshold sweep
# Step 1: quick sanity check with 50 samples
from src.evaluation.eval_router import run_router_sweep

sweep_df = run_router_sweep(experiment_name='exp6_router', max_samples=50)
print(sweep_df[['threshold', 'accuracy', 'mean_token_count', 'pct_easy']].to_string(index=False))

In [ ]:
# Exp 6 — full sweep (set max_samples: null in eval.yaml for full 1319-sample eval)
# After confirming the sanity check above looks reasonable, run:
sweep_df_full = run_router_sweep(experiment_name='exp6_router_full', max_samples=None)
print(sweep_df_full[['threshold', 'accuracy', 'mean_token_count', 'pct_easy']].to_string(index=False))